In [1]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import gc

from utilities import walk_through_files, get_diff_stringval_from_2columns, auto_reduce_mem

pd.set_option('display.max_rows', 250)
pd.set_option('display.max_columns', 150)

In [2]:
FP_CDGRAVITY = os.path.join('intermediate-files', 'daily-movecount.csv.gz')

FP_WEATHERSTATION_CROSSWALK = os.path.join('intermediate-files', 'subzones2014-weatherstation-crosswalk.csv')
FP_WEATHER = os.path.join('..', 'raw-files', 'SWEAT', 'datafiles', 'weather')
# FP_WEATHERSTATION_ID = os.path.join('..', 'raw-files', 'SWEAT', 'datafiles', 'station-data.csv')

FP_CD_AREANAMES = os.path.join('intermediate-files', 'citydata-subzones.csv')
FP_HTYPE = os.path.join('intermediate-files', 'htype.csv')
FP_SUBZONES_SHP = os.path.join('zip:/' '..', 'raw-files', 'master-plan-2014-subzone-boundary-no-sea', 'master-plan-2014-subzone-boundary-no-sea-shp.zip')

FP_REALIS = os.path.join('intermediate-files', 'realis_with_postal_code_complete.csv.gz')
FP_HDBRESALE = os.path.join('intermediate-files', 'hdb_saledata_with_geodata.csv.gz')

FP_AGEGENDER = os.path.join('..', 'raw-files', 'resident-population-by-planning-area-subzone-age-group-and-sex-2015', 'resident-population-by-subzone-age-group-and-sex.csv')
FP_ETHNIC = os.path.join('..', 'raw-files', 'resident-population-by-planning-area-subzone-ethnic-group-and-sex-2015', 'resident-population-by-subzone-ethnic-group-and-sex.csv')

FP_DIST = os.path.join('intermediate-files', 'distance_subzone_2014.csv')
FP_POI = os.path.join('intermediate-files', 'subzones-poi-count.csv')

FP_SUBZONES_SHP = os.path.join('zip://', '..', 'raw-files', 
                               'master-plan-2014-subzone-boundary-no-sea', 'master-plan-2014-subzone-boundary-no-sea-shp.zip')
# import fiona
# fiona.drvsupport.supported_drivers['KML'] = 'rw' # enable KML support which is disabled by default
SAVEPATH = os.path.join('..', 'analysis', 'cd-panel.csv')

### Prep Citydata gravity panel

In [3]:
# Prep area to xid dict AND dict to clean 4 names
xid_area = (pd.read_csv(FP_CD_AREANAMES)
            .rename(columns={'subzone': 'area', 'areaid': 'area_id'})
            .assign(area=lambda df: df['area'].str.lower().str.strip())
           )
xid_area = dict(zip(xid_area['area_id'], xid_area['area']))

# Clean obvious name errors from Citydata (to get perfect match)
clean_cd_subzone = {'marina east mp': 'marina east (mp)', 
                    'monks hill': "monk's hill", 
                    "national university of s pore": "national university of s'pore", 
                    "peoples park": "people's park"}

In [4]:
# Read in 
df = (pd.read_csv(FP_CDGRAVITY, compression='gzip'))

df = (pd.read_csv(FP_CDGRAVITY, compression='gzip')
      .merge((df.groupby(['home', 'date'])['inflow_count'].sum().reset_index()
              .rename(columns={'inflow_count': 'basecount'})), 
             how='left', on=['home', 'date']
            )
      .merge((df.groupby(['area_id', 'date'])['inflow_count'].sum().reset_index()
              .rename(columns={'inflow_count': 'dest_basecount'})), 
             how='left', on=['area_id', 'date']
            )
      .rename(columns={'inflow_count': 'inflow'})
      .assign(inflow=lambda df: df['inflow']/df['basecount']*100)
      .assign(date=lambda df: df['date']+100000)
      .assign(date=lambda df: pd.to_datetime(df['date'],unit='s'))
      .assign(date=lambda df: [timestamp.date() for timestamp in df['date']])
      .assign(home=lambda df: df['home'].map(xid_area))
      .assign(area=lambda df: df['area_id'].map(xid_area))
      .drop('area_id', axis=1)
      .assign(home=lambda df: df['home'].replace(clean_cd_subzone))
      .assign(area=lambda df: df['area'].replace(clean_cd_subzone))
     )

del xid_area
gc.collect()

0

### Merge with house types

In [5]:
# read in and merge
htype = (pd.read_csv(FP_HTYPE)
         .assign(area=lambda df: df['area'].str.lower().str.strip())
        )
# Get home area dwelling type
df = df.merge(htype.add_suffix('_home').rename(columns={'area_home': 'home'}),
              how='left', on='home', validate='m:1')
# Get dest area dwelling type
df = df.merge(htype.add_suffix('_dest').rename(columns={'area_dest': 'area'}),
              how='left', on='area', validate='m:1')
del htype
gc.collect()

53

### Prep Weather

In [6]:
# 2014 Area to weather station crosswalk
ws_area_crosswalk = (pd.read_csv(FP_WEATHERSTATION_CROSSWALK)
                     .rename(columns={'subzone': 'area', 'weather_station': 'wsid'})
                     .assign(area=lambda df: df['area'].str.lower().str.strip())
                    )

ws_area_crosswalk = dict(zip(ws_area_crosswalk['area'], ws_area_crosswalk['wsid']))

In [7]:
# Read in weather data
WEATHER_DATA = list(walk_through_files(FP_WEATHER, 'csv', returnboth=True))

weather = pd.DataFrame()

_usecols = ['Year', 'Month', 'Day', 'Daily Rainfall Total (mm)']
for (path, file) in WEATHER_DATA:
    for period in ['202001', '202002', '202003']:
        if period not in file:
            continue
            
        wsid = file.split('-')[0]
        
        _df = (pd.read_csv(os.path.join(path,file), usecols=_usecols)
               .assign(wsid=wsid)
               .rename(columns={'Year': 'year', 'Month':'month', 'Day':'day', 'Daily Rainfall Total (mm)': 'rainfall'})
               .assign(date=lambda df: pd.to_datetime(df[['year', 'month', 'day']]))
               .assign(date=lambda df: [timestamp.date() for timestamp in df['date']])
               .drop(['year', 'month', 'day'], axis=1)
              )
        
        weather = weather.append(_df)

In [8]:
len(weather['wsid'].unique().tolist())

46

In [9]:
# weather = (weather.merge(ws_area_crosswalk, how='inner', on='wsid', validate='m:m')
#            .drop('wsid', axis=1)
#           )

In [10]:
# Fill in missing data
weather = weather.groupby(['wsid']).apply(lambda group: group.interpolate(method='linear'))

### Merge with weather

In [11]:
assert type(df.loc[0, 'date']) ==type(weather.reset_index().loc[0, 'date'])

In [12]:
df['wsid_home'] = df['home'].map(ws_area_crosswalk)
df['wsid_dest'] = df['area'].map(ws_area_crosswalk)

In [13]:
# Home area weather
df = df.merge(weather.rename(columns={'wsid': 'wsid_home', 'rainfall': 'rain_home'}), 
         how='left', on=['date', 'wsid_home'], validate='m:1')

# Dest area weather
df = df.merge(weather.rename(columns={'wsid': 'wsid_dest', 'rainfall': 'rain_dest'}), 
         how='left', on=['date', 'wsid_dest'], validate='m:1')

del weather
gc.collect()

df.drop(['wsid_home', 'wsid_dest'], axis=1, inplace=True)

### Merge area distance and continguity

In [14]:
_usecols = ['Subzone Name_o', 'Subzone Name_d',
            'centroid_distance', 'edge_distance', 'centroid_distance_epsg', 'edge_distance_epsg']

dist = (pd.read_csv(FP_DIST, usecols=_usecols)
        .rename(columns={'Subzone Name_o':'home', 'Subzone Name_d': 'area'})
        .assign(home=lambda df: df['home'].str.lower().str.strip())
        .assign(area=lambda df: df['area'].str.lower().str.strip())
       )

df = df.merge(dist, how='left', on=['home', 'area'], validate='m:m')

del dist
gc.collect()

0

In [15]:
# get_diff_stringval_from_2columns(df, 'home', dist, 'home')

In [16]:
# get_diff_stringval_from_2columns(df, 'area', dist, 'area')

### Get area's land area

In [17]:
# https://www.singstat.gov.sg/-/media/files/publications/population/population2018-map1.pdf
subzones_area = (gpd.read_file(FP_SUBZONES_SHP)
                 .rename(columns={'SUBZONE_N': 'subzone', 'REGION_C': 'region'})
                 .assign(subzone=lambda gdf: gdf['subzone'].str.lower())
                 .assign(area=lambda gdf: gdf['geometry'].area / (10**6))
                 .filter(['subzone', 'area', 'region'])
                )
# subzones_area.crs

In [18]:
# Merge
df = df.merge(subzones_area.rename(columns={'subzone':'home', 'area': 'area_o', 'region': 'region_o'}), 
              how='left', on='home', validate='m:1')

df = df.merge(subzones_area.rename(columns={'subzone':'area', 'area': 'area_d', 'region': 'region_d'}), 
              how='left', on='area', validate='m:1')

### Merge other available subzone census var.

#### Age and gender

In [19]:
# Read in and prep
collapse_age_brackets = {'0 - 4': '0-9', 
                         '5 - 9': '0-9', 
                         '10 - 14': '10-19', 
                         '15 - 19': '10-19', 
                         '20 - 24': '20-39',
                         '25 - 29': '20-39',
                         '30 - 34': '20-39', 
                         '35 - 39': '20-39',
                         '40 - 44': '40-64',
                         '45 - 49': '40-64',
                         '50 - 54': '40-64',
                         '55 - 59': '40-64',
                         '60 - 64': '40-64',
                         '65 - 69': '>65',
                         '70 - 74': '>65',
                         '75 - 79': '>65',
                         '80 - 84': '>65',
                         '85 & Over': '>65'}
         
age_bracket_codes = {'0-9': 1, 
                     '10-19': 2, 
                     '20-39': 3, 
                     '40-64': 4, 
                     '>65':5}

# Agegender
agegender = (pd.read_csv(FP_AGEGENDER, usecols=['level_1', 'level_2', 'level_4', 'value'])
             .rename(columns={'level_1': 'gender', 'level_2': 'age', 'level_4': 'subzone'})
             .query('age!="Total"').query('gender!="Total"')
             .assign(subzone=lambda df: df['subzone'].str.lower())
             .assign(age=lambda df: df['age'].map(collapse_age_brackets))
             .assign(age=lambda df: df['age'].map(age_bracket_codes))
            )

ttpop = (agegender.groupby('subzone')['value'].sum().reset_index()           
         .rename(columns={'value': 'tt_pop'})
        )

females = (agegender.query('gender=="Females"')
           .drop('gender', axis=1)
           .groupby('subzone')['value'].sum().reset_index()
           .rename(columns={'value': 'females'})
          )


age_20 = (agegender.query('age<=2')
          .drop('age', axis=1)
          .groupby(['subzone']).sum().reset_index()
          .rename(columns={'value': 'age_20'})
         )
age20_40 = (agegender.query('age==3')
            .drop('age', axis=1)
            .groupby(['subzone']).sum().reset_index()
            .rename(columns={'value': 'age20_40'})
          )

age65_ = (agegender.query('age==5')
          .drop('age', axis=1)
          .groupby(['subzone']).sum().reset_index()
          .rename(columns={'value': 'age65_'})
         )

In [20]:
# Merge origin
for pd_df in [females, age_20, age20_40, age65_, ttpop]:
    
    _pd_df = pd_df.add_suffix('_o').rename(columns={'subzone_o':'home'})
    
    df = df.merge(_pd_df, how='left', on='home', validate='m:1')

# Merge dest
for pd_df in [females, age_20, age20_40, age65_, ttpop]:
    
    _pd_df = pd_df.add_suffix('_d').rename(columns={'subzone_d':'area'})
    
    df = df.merge(_pd_df, how='left', on='area', validate='m:1')
    
del agegender, females, ttpop,age20_40, age_20, age65_

In [21]:
# Get proportions and drop total
for x in ['females_o', 'age_20_o', 'age20_40_o', 'age65__o']:
    df[x] = 100 * (df[x] / df['tt_pop_o'])

for x in ['females_d', 'age_20_d', 'age20_40_d', 'age65__d']:
    df[x] = 100 * (df[x] / df['tt_pop_d'])    
    
df.drop(['tt_pop_o', 'tt_pop_d'], axis=1, inplace=True)    

#### Ethnic

In [22]:
# Read in and prep
ethnic = (pd.read_csv(FP_ETHNIC, usecols=['level_1', 'level_2', 'level_4', 'value'])
          .rename(columns={'level_1': 'ethnic', 'level_2':'sex', 'level_4': 'subzone'})
          .query('ethnic!="Total"')
          .query('sex!="Total"').drop('sex', axis=1)
          .assign(subzone=lambda df: df['subzone'].str.lower().str.strip())
         )

chinese = (ethnic[ethnic['ethnic']=='Chinese']
           .drop('ethnic', axis=1)
           .groupby('subzone').sum().reset_index()
           .rename(columns={'value': 'chinese'})
          )

malay = (ethnic[ethnic['ethnic']=='Malays']
         .drop('ethnic', axis=1)
         .groupby('subzone').sum().reset_index()
         .rename(columns={'value': 'malays'})
        )

indian = (ethnic[ethnic['ethnic']=='Indians']
           .drop('ethnic', axis=1)
           .groupby('subzone').sum().reset_index()
           .rename(columns={'value': 'indians'})
          )

other = (ethnic[ethnic['ethnic']=='Others']
           .drop('ethnic', axis=1)
           .groupby('subzone').sum().reset_index()
           .rename(columns={'value': 'others'})
          )

tt_ethnic = (ethnic
             .drop('ethnic', axis=1)
             .groupby('subzone').sum().reset_index()
             .rename(columns={'value': 'tt_ethnic'})
            )

In [23]:
# Merge origin
for pd_df in [chinese, malay, indian, other, tt_ethnic]:
    
    _pd_df = pd_df.add_suffix('_o').rename(columns={'subzone_o':'home'})
    
    df = df.merge(_pd_df, how='left', on='home', validate='m:1')

# Merge dest
for pd_df in [chinese, malay, indian, other, tt_ethnic]:
    
    _pd_df = pd_df.add_suffix('_d').rename(columns={'subzone_d':'area'})
    
    df = df.merge(_pd_df, how='left', on='area', validate='m:1')
    
del chinese, indian, malay, other, tt_ethnic

In [24]:
# Get proportions and drop total
for x in ['chinese_o', 'malays_o', 'indians_o', 'others_o']:
    df[x] = 100 * (df[x] / df['tt_ethnic_o'])

for x in ['chinese_d', 'malays_d', 'indians_d', 'others_d']:
    df[x] = 100 * (df[x] / df['tt_ethnic_d'])    
    
df.drop(['tt_ethnic_o', 'tt_ethnic_d'], axis=1, inplace=True)    

### Merge with POIs count

In [25]:
poi = (pd.read_csv(FP_POI)
       .assign(subzone=lambda df: df['subzone'].str.lower().str.strip())
       .rename(columns={'subzone':'area'})
      )

df = df.merge(poi, how='left', on='area', validate='m:1')

del poi
gc.collect()

147

### Merge Public Rentals

In [26]:
FP_PUB_RENTALS = os.path.join('intermediate-files', 'subzone_rental_portion.csv')

In [27]:
_usecols = ['area', '1room_rental_pct', '2room_rental_pct', '3room_rental_pct', 'other_room_rental_pct', 'total_dwelling_units']
pub_rentals = (pd.read_csv(FP_PUB_RENTALS, usecols=_usecols)
               .query('total_dwelling_units!=0')
               .assign(rm12_rental_pct=lambda df: df['1room_rental_pct'] + df['2room_rental_pct'])
               .assign(rental_pct=lambda df: df['rm12_rental_pct'] + df['3room_rental_pct'] + df['other_room_rental_pct'])
               .drop(['1room_rental_pct', '2room_rental_pct', '3room_rental_pct', 'other_room_rental_pct'], axis=1)
              )

df = df.merge(pub_rentals.add_suffix('_home').rename(columns={'area_home': 'home'}),
              how='left', on='home', validate='m:1')

df = df.merge(pub_rentals.add_suffix('_dest').rename(columns={'area_dest': 'area'}),
              how='left', on='area', validate='m:1')

del pub_rentals
gc.collect()

16

### Merge resale price

#### Avg public & private together

In [28]:
YEARS = [2017, 2018, 2019] # 3 years before sample year of 2020

df_price = ((pd.read_csv(FP_HDBRESALE, compression='gzip', usecols=['month', 'psm', 'resale_price', 'subzone_name'])
             .rename(columns={'resale_price': 'price'})
             .assign(area=lambda df: df['subzone_name'].str.lower())
             # retain only transactions in 2017 to 2019
             .assign(year=lambda df: df['month'].str.slice(stop=4).astype(int))
             .query('year in @YEARS')
             .filter(['area', 'psm', 'price'])
            ).append((pd.read_csv(FP_REALIS, compression='gzip', usecols=['contractdate', 'unitpricepsm', 'transactedprice', 'subzone_name'])
                      .rename(columns={'transactedprice': 'price', 'unitpricepsm': 'psm'})
                      .assign(area=lambda df: df['subzone_name'].str.lower())
                      # retain only transactions in 2017 to 2019
                      .assign(year=lambda df: df['contractdate'].str.slice(stop=4).astype(int))
                      .query('year in @YEARS')
                      .filter(['area', 'psm', 'price'])
                     )
                    )
            # Remove comma from valuations (e.g. 17,493)
            .assign(psm=lambda df: df['psm'].astype('string').str.replace(',','').astype('float'))
            .assign(price=lambda df: df['price'].astype('string').str.replace(',','').astype('float'))
            # Get num of property transactions by subzone
            .pipe(lambda df: df.merge(df.groupby('area').size().reset_index()
                                      .rename(columns={0: 'num_transactions'}), 
                                      how='left', on='area', validate='m:1'))
            .groupby(['area', 'num_transactions']).mean().reset_index()
           )

df_price.head()

D:\ProgramData\Anaconda3\envs\gis\lib\site-packages\IPython\core\interactiveshell.py:3170: DtypeWarning: Columns (5,6) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


,area,num_transactions,psm,price
0,admiralty,374.0,3930.897154,3.698096e+05
1,alexandra hill,184.0,5493.879995,5.314567e+05
2,alexandra north,733.0,18564.409277,1.697418e+06
3,aljunied,2311.0,12530.475712,1.034765e+06
4,anak bukit,909.0,13255.160067,1.720123e+06


In [29]:
df = df.merge(df_price.add_suffix('_home').rename(columns={'area_home': 'home'}),
              how='left', on='home', validate='m:1')

df = df.merge(df_price.add_suffix('_dest').rename(columns={'area_dest': 'area'}),
              how='left', on='area', validate='m:1')

In [30]:
df_price.num_transactions.sum()

133035.0

#### Keep public & private separate

In [31]:
df_hdb_price = ((pd.read_csv(FP_HDBRESALE, compression='gzip',  usecols=['month', 'psm', 'resale_price', 'subzone_name'])
                 .rename(columns={'resale_price': 'price'})
                 .assign(area=lambda df: df['subzone_name'].str.lower())
                 # retain only transactions in 2017 to 2019
                 .assign(year=lambda df: df['month'].str.slice(stop=4).astype(int))
                 .query('year in @YEARS')
                 .filter(['area', 'psm', 'price'])
                 # Get num of property transactions by subzone
                 .pipe(lambda df: df.merge(df.groupby('area').size().reset_index()
                                           .rename(columns={0: 'public_num_transactions'}), 
                                           how='left', on='area', validate='m:1'))
                )
                # Take avg of psm over areas
                .groupby(['area', 'public_num_transactions']).mean().reset_index()
                .rename(columns={'psm': 'public_psm', 'price': 'public_price'})
               )

# Add to df_price (subzone level transaction price)
df_price = df_price.merge(df_hdb_price, how='left', on='area', validate='1:1')

df_hdb_price.head(2)

,area,public_num_transactions,public_psm,public_price
0,admiralty,374,3930.897154,369809.569519
1,alexandra hill,184,5493.879995,531456.739130


In [32]:
# Merge with df
df = df.merge(df_hdb_price.add_suffix('_home').rename(columns={'area_home': 'home'}),
              how='left', on='home', validate='m:1')

df = df.merge(df_hdb_price.add_suffix('_dest').rename(columns={'area_dest': 'area'}),
              how='left', on='area', validate='m:1')

del df_hdb_price
gc.collect()

68

In [33]:
df_private_price = ((pd.read_csv(FP_REALIS, compression='gzip', usecols=['contractdate', 'unitpricepsm', 'transactedprice', 'subzone_name'])
                     .rename(columns={'transactedprice': 'price', 'unitpricepsm': 'psm'})
                     .assign(area=lambda df: df['subzone_name'].str.lower())
                     # retain only transactions in 2017 to 2019
                     .assign(year=lambda df: df['contractdate'].str.slice(stop=4).astype(int))
                     .query('year in @YEARS')
                     .filter(['area', 'psm', 'price'])
                     # Remove comma from valuations (e.g. 17,493)
                     .assign(psm=lambda df: df['psm'].astype('string').str.replace(',','').astype('float'))
                     .assign(price=lambda df: df['price'].astype('string').str.replace(',','').astype('float'))
                     .pipe(lambda df: df.merge(df.groupby('area').size().reset_index()
                                               .rename(columns={0: 'private_num_transactions'}), 
                                               how='left', on='area', validate='m:1'))
                     )
                    # Take avg of psm over areas
                    .groupby(['area', 'private_num_transactions']).mean().reset_index()
                    .rename(columns={'psm': 'private_psm', 'price': 'private_price'})
                   )

# Add to df_price (subzone level transaction price)
df_price = df_price.merge(df_private_price, how='left', on='area', validate='1:1')

df_private_price.head(2)

D:\ProgramData\Anaconda3\envs\gis\lib\site-packages\IPython\core\interactiveshell.py:3170: DtypeWarning: Columns (5,6) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


,area,private_num_transactions,private_psm,private_price
0,alexandra north,733.0,18564.409277,1.697418e+06
1,aljunied,1869.0,14140.758159,1.152435e+06


In [34]:
# Merge with df
df = df.merge(df_private_price.add_suffix('_home').rename(columns={'area_home': 'home'}),
              how='left', on='home', validate='m:1')

df = df.merge(df_private_price.add_suffix('_dest').rename(columns={'area_dest': 'area'}),
              how='left', on='area', validate='m:1')

del df_private_price
gc.collect()

87

In [35]:
# Save subzone level transaction price
df_price.to_csv(os.path.join('intermediate-files', 'propprice-subzone.csv'), index=False)

del df_price
gc.collect()

22

#### Compute weighted (by public/private residence share) house price

In [36]:
# weighted PSM for home
df['wpsm_private_home'] = df['private_home']/100 * df['private_psm_home']
df['wpsm_public_home'] = (100 - df['private_home'])/100 * df['public_psm_home']
df['wpsm_home'] = df[['wpsm_private_home', 'wpsm_public_home']].sum(axis=1, skipna=True) # Treat na as 0
# If zero, means both private and public data is missing, replace as missing
df['wpsm_home'] = np.where(df['wpsm_home']==0, np.nan, df['wpsm_home'])

# weighted PSM for dest
df['wpsm_private_dest'] = df['private_dest']/100 * df['private_psm_dest']
df['wpsm_public_dest'] = (100 - df['private_dest'])/100 * df['public_psm_dest']
df['wpsm_dest'] = df[['wpsm_private_dest', 'wpsm_public_dest']].sum(axis=1, skipna=True) # Treat na as 0
# If zero, means both private and public data is missing, replace as missing
df['wpsm_dest'] = np.where(df['wpsm_dest']==0, np.nan, df['wpsm_dest'])

# weighted price for home
df['wprice_private_home'] = df['private_home']/100 * df['private_price_home']
df['wprice_public_home'] = (100 - df['private_home'])/100 * df['public_price_home']
df['wprice_home'] = df[['wprice_private_home', 'wprice_public_home']].sum(axis=1, skipna=True) # Treat na as 0
# If zero, means both private and public data is missing, replace as missing
df['wprice_home'] = np.where(df['wprice_home']==0, np.nan, df['wprice_home'])

# weighted price for dest
df['wprice_private_dest'] = df['private_home']/100 * df['private_price_dest']
df['wprice_public_dest'] = (100 - df['private_home'])/100 * df['public_price_dest']
df['wprice_dest'] = df[['wprice_private_dest', 'wprice_public_dest']].sum(axis=1, skipna=True) # Treat na as 0 
# If zero, means both private and public data is missing, replace as missing
df['wprice_dest'] = np.where(df['wprice_dest']==0, np.nan, df['wprice_dest'])

### Save

In [37]:
# Dont remember what this is for
_g = ['pa_home', 'home', 'rm12_home']
df[_g].drop_duplicates().sort_values('rm12_home', ascending=False).head(10)

,pa_home,home,rm12_home
682,Kallang,crawford,49.208145
5,Rochor,sungei road,32.835821
2910,Toa Payoh,boon teck,32.216679
1925,Bukit Merah,henderson hill,29.372197
60,Bukit Merah,alexandra hill,29.288390
1846,Kallang,lavender,27.233115
342,Bukit Merah,bukit ho swee,25.084746
2925,Queenstown,commonwealth,23.162275
2941,Bukit Merah,kampong tiong bahru,22.489083
2978,Bukit Merah,telok blangah way,20.209424


In [38]:
# Codify area names (remove text names, e.g. Crawford --> 16)
for col in ['home', 'area', 'pa_home', 'pa_dest']:
    df[col] = df[col].astype('category').cat.codes

In [39]:
df.head(3)

,date,home,inflow,basecount,dest_basecount,area,pa_home,resident_count_home,private_home,rm12_home,rm3_home,rm4_home,rm5_home,pa_dest,resident_count_dest,private_dest,rm12_dest,rm3_dest,rm4_dest,rm5_dest,rain_home,rain_dest,centroid_distance,edge_distance,centroid_distance_epsg,edge_distance_epsg,area_o,region_o,area_d,region_d,females_o,age_20_o,age20_40_o,age65__o,females_d,age_20_d,age20_40_d,age65__d,chinese_o,malays_o,indians_o,others_o,chinese_d,malays_d,indians_d,others_d,libraries,supermarkets,parks,preschools,schzones,silverzones,sport_facilities,transit,tour,schools,mrts,lrts,manufacturing,construction,services,total_dwelling_units_home,rm12_rental_pct_home,rental_pct_home,total_dwelling_units_dest,rm12_rental_pct_dest,rental_pct_dest,num_transactions_home,psm_home,price_home,num_transactions_dest,psm_dest,price_dest,public_num_transactions_home,public_psm_home,public_price_home,public_num_transactions_dest,public_psm_dest,public_price_dest,private_num_transactions_home,private_psm_home,private_price_home,private_num_transactions_dest,private_psm_dest,private_price_dest,wpsm_private_home,wpsm_public_home,wpsm_home,wpsm_private_dest,wpsm_public_dest,wpsm_dest,wprice_private_home,wprice_public_home,wprice_home,wprice_private_dest,wprice_public_dest,wprice_dest
0,2020-01-01,0,14.583333,48,24,0,36,14110,0.0,8.079376,5.669738,42.168675,44.082211,36,14110,0.0,8.079376,5.669738,42.168675,44.082211,0.0,0.0,0.000000,0.000000,0.000000,0.000000,1.261649,NR,1.261649,NR,50.589868,27.064539,31.158917,7.841777,50.589868,27.064539,31.158917,7.841777,71.111111,15.902778,9.236111,3.75,71.111111,15.902778,9.236111,3.75,0,2,0,9,1,0,0,0,0,1,0,0,4,10,148,4479.0,0.05269,0.05269,4479.0,0.05269,0.05269,374.0,3930.897154,3.698096e+05,374.0,3930.897154,369809.569519,374.0,3930.897154,369809.569519,374.0,3930.897154,369809.569519,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3930.897154,3930.897154,NaN,3930.897154,3930.897154,NaN,369809.569519,3.698096e+05,NaN,369809.569519,369809.569519
1,2020-01-01,37,0.019914,15065,24,0,12,200,100.0,0.000000,0.000000,0.000000,0.000000,36,14110,0.0,8.079376,5.669738,42.168675,44.082211,0.0,0.0,0.176347,0.165162,17009.925465,15966.639935,0.196620,CR,1.261649,NR,42.857143,7.142857,50.000000,NaN,50.589868,27.064539,31.158917,7.841777,56.250000,6.250000,12.500000,25.00,71.111111,15.902778,9.236111,3.75,0,2,0,9,1,0,0,0,0,1,0,0,4,10,148,NaN,NaN,NaN,4479.0,0.05269,0.05269,166.0,22025.536145,1.889564e+06,374.0,3930.897154,369809.569519,NaN,NaN,NaN,374.0,3930.897154,369809.569519,166.0,22025.536145,1.889564e+06,NaN,NaN,NaN,22025.536145,NaN,22025.536145,NaN,3930.897154,3930.897154,1.889564e+06,NaN,1.889564e+06,NaN,0.000000,NaN
2,2020-01-01,105,0.735294,136,24,0,15,440,100.0,0.000000,0.000000,0.000000,0.000000,36,14110,0.0,8.079376,5.669738,42.168675,44.082211,0.0,0.0,0.145839,0.135053,15652.302507,14512.911181,0.554624,WR,1.261649,NR,NaN,NaN,NaN,NaN,50.589868,27.064539,31.158917,7.841777,NaN,NaN,NaN,NaN,71.111111,15.902778,9.236111,3.75,0,2,0,9,1,0,0,0,0,1,0,0,4,10,148,NaN,NaN,NaN,4479.0,0.05269,0.05269,44.0,18756.772727,1.109993e+06,374.0,3930.897154,369809.569519,NaN,NaN,NaN,374.0,3930.897154,369809.569519,44.0,18756.772727,1.109993e+06,NaN,NaN,NaN,18756.772727,NaN,18756.772727,NaN,3930.897154,3930.897154,1.109993e+06,NaN,1.109993e+06,NaN,0.000000,NaN


In [40]:
df.to_csv(SAVEPATH, index=False)